# 02: 日次全件スナップショットの最適化（Upsert と SCD Type 2）

## このノートブックで学習できること

- 毎日全件ダンプされる `crm_customers.csv` を「日付パーティションで毎日全件積み上げ」する従来のアンチパターンで、容量がどれだけ重複膨張するかのシミュレーション
- 解決策1（SCD Type 1 Upsert）: `customer_id` を主キーとして変更行と新規行だけを反映する実装
- 解決策2（SCD Type 2 履歴管理）: `valid_from`、`valid_to`、`is_current` を持たせ、会員ランク（`membership_tier`）などが変化した顧客だけ新バージョン行を追加して過去属性を保持する実装

## 到達ゴール

- 日次全件連携データに対して、変更のない大部分のデータを重複保存せずに最新状態（SCD Type 1）または履歴付き状態（SCD Type 2）で管理するパターンを使い分けられる

## なぜこれが必要か（従来の BigQuery / Hive 運用との比較）

従来の DWH や Hive 運用では、日次全件連携のマスターデータを `dt=2026-11-01`, `dt=2026-11-02` のように日付パーティションへ毎日コピー保存する設計がよく採られていました。しかし前日からの変更が数%しかない場合、9割以上の重複データが毎日積み上がり、ストレージコストとスキャン量が線形に膨張します。Iceberg の `upsert`（MERGE）や SCD Type 2 を使うことで、変更分だけを効率よく反映できます。


## 1. アンチパターン: 日次全件パーティション積み上げの膨張シミュレーション

このセルでは、Day 1〜Day 3 の `crm_customers.csv`（計 500 + 515 + 525 = 1,540 行）を、重複を排除せずそのまま日付パーティション付きの Iceberg テーブルへ毎日追記した場合の行数と物理データ容量を確認します。


In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.catalog_helper import (
    LANDING_DIR,
    LAKEHOUSE_DIR,
    get_duckdb_conn,
    get_iceberg_catalog,
    init_storage,
    register_iceberg_table,
)
import polars as pl
import pyarrow as pa
from pyiceberg.schema import Schema
from pyiceberg.types import BooleanType, DateType, NestedField, StringType

catalog = get_iceberg_catalog()

def dir_parquet_bytes(table) -> int:
    data_dir = Path(table.location().replace("file://", "")) / "data"
    return sum(p.stat().st_size for p in data_dir.rglob("*.parquet"))

if catalog.table_exists("bronze.crm_customers_antipattern"):
    catalog.drop_table("bronze.crm_customers_antipattern")

frames = []
for day, dt in [("day1", "2026-11-01"), ("day2", "2026-11-02"), ("day3", "2026-11-03")]:
    df = pl.read_csv(LANDING_DIR / day / "crm_customers.csv").with_columns(pl.lit(dt).alias("snapshot_dt"))
    frames.append(df)

anti_arrow = pl.concat(frames).to_arrow()
anti_tbl = catalog.create_table("bronze.crm_customers_antipattern", schema=anti_arrow.schema)
anti_tbl.append(anti_arrow)

print(f"アンチパターン総行数   : {anti_tbl.scan().to_arrow().num_rows} 行")
print(f"アンチパターン総容量   : {dir_parquet_bytes(anti_tbl):,} bytes")


3 日分だけで総行数が 1,540 行に膨らみました。実際には 525 人しか顧客がいないため、約 1,000 行以上が前日と同じデータの重複保存です。

## 2. 解決策 1: SCD Type 1（Upsert による最新状態の維持）

次のセルでは、`customer_id` を識別子（`identifier_field_ids`）に設定した Iceberg テーブル `silver.crm_customers_scd1` を作成し、Day 1 → Day 2 → Day 3 の順に `upsert()` を実行して最新状態だけを保持します。


In [ ]:
if catalog.table_exists("silver.crm_customers_scd1"):
    catalog.drop_table("silver.crm_customers_scd1")

d1_df = pl.read_csv(LANDING_DIR / "day1" / "crm_customers.csv")
scd1_schema = Schema(
    NestedField(1, "customer_id", StringType(), required=True),
    NestedField(2, "name", StringType(), required=True),
    NestedField(3, "email", StringType(), required=True),
    NestedField(4, "phone", StringType(), required=True),
    NestedField(5, "birth_date", StringType(), required=True),
    NestedField(6, "membership_tier", StringType(), required=True),
    NestedField(7, "status", StringType(), required=True),
    NestedField(8, "updated_at", StringType(), required=True),
    identifier_field_ids=[1],
)
scd1_tbl = catalog.create_table("silver.crm_customers_scd1", schema=scd1_schema)
pa_scd1_schema = scd1_tbl.schema().as_arrow()

for day in ["day1", "day2", "day3"]:
    df_day = pl.read_csv(LANDING_DIR / day / "crm_customers.csv")
    res = scd1_tbl.upsert(pa.Table.from_pylist(df_day.to_dicts(), schema=pa_scd1_schema))
    print(f"{day} upsert -> updated={res.rows_updated}, inserted={res.rows_inserted}")

print(f"SCD Type 1 最終行数: {scd1_tbl.scan().to_arrow().num_rows} 行")


Day 2 では変更のあった 25 件が `updated`、新規会員 15 件が `inserted` となり、Day 3 でも変更 5 件・新規 10 件だけが反映されて最終行数は `525` 行に抑えられました。

## 3. 解決策 2: SCD Type 2（変化した顧客のみ履歴を保持）

SCD Type 1 では過去の会員ランクが上書きされます。「購入時点の会員ランクで過去売上を分析したい」場合は SCD Type 2 を採用します。次のセルでは、`valid_from`・`valid_to`・`is_current` を持つ `silver.crm_customers_scd2` テーブルを構築し、Day 1〜Day 3 で変化があった顧客だけ旧レコードをクローズして新バージョン行を追加します。


In [ ]:
if catalog.table_exists("silver.crm_customers_scd2"):
    catalog.drop_table("silver.crm_customers_scd2")

scd2_schema = Schema(
    NestedField(1, "customer_id", StringType(), required=True),
    NestedField(2, "name", StringType(), required=True),
    NestedField(3, "email", StringType(), required=True),
    NestedField(4, "membership_tier", StringType(), required=True),
    NestedField(5, "status", StringType(), required=True),
    NestedField(6, "valid_from", StringType(), required=True),
    NestedField(7, "valid_to", StringType(), required=False),
    NestedField(8, "is_current", BooleanType(), required=True),
)
scd2_tbl = catalog.create_table("silver.crm_customers_scd2", schema=scd2_schema)
pa_scd2_schema = scd2_tbl.schema().as_arrow()

def apply_scd2_day(day: str, effective_date: str) -> None:
    incoming = pl.read_csv(LANDING_DIR / day / "crm_customers.csv").select(
        ["customer_id", "name", "email", "membership_tier", "status"]
    )
    existing = pl.from_arrow(scd2_tbl.scan().to_arrow())
    if existing.height == 0:
        init_df = incoming.with_columns([
            pl.lit(effective_date).alias("valid_from"),
            pl.lit(None, dtype=pl.String).alias("valid_to"),
            pl.lit(True).alias("is_current"),
        ])
        scd2_tbl.overwrite(pa.Table.from_pylist(init_df.to_dicts(), schema=pa_scd2_schema))
        return

    current = existing.filter(pl.col("is_current"))
    history = existing.filter(~pl.col("is_current"))

    joined = incoming.join(current, on="customer_id", how="left", suffix="_old")
    unchanged_ids = joined.filter(
        (pl.col("membership_tier") == pl.col("membership_tier_old"))
        & (pl.col("status") == pl.col("status_old"))
    )["customer_id"]

    # 変化なしはそのまま維持、変化ありは旧版をクローズして新版を追加
    kept_current = current.filter(pl.col("customer_id").is_in(unchanged_ids))
    closed_old = current.filter(~pl.col("customer_id").is_in(unchanged_ids)).with_columns([
        pl.lit(effective_date).alias("valid_to"),
        pl.lit(False).alias("is_current"),
    ])
    new_or_changed = incoming.filter(~pl.col("customer_id").is_in(unchanged_ids)).with_columns([
        pl.lit(effective_date).alias("valid_from"),
        pl.lit(None, dtype=pl.String).alias("valid_to"),
        pl.lit(True).alias("is_current"),
    ])

    next_state = pl.concat([history, kept_current, closed_old, new_or_changed])
    scd2_tbl.overwrite(pa.Table.from_pylist(next_state.to_dicts(), schema=pa_scd2_schema))

apply_scd2_day("day1", "2026-11-01")
apply_scd2_day("day2", "2026-11-02")
apply_scd2_day("day3", "2026-11-03")

res_scd2 = pl.from_arrow(scd2_tbl.scan().to_arrow())
print(f"SCD Type 2 総行数: {res_scd2.height} 行 (うち is_current=True: {res_scd2.filter(pl.col('is_current')).height} 行)")
changed_sample = res_scd2.filter(pl.col("customer_id").is_in(
    res_scd2.group_by("customer_id").len().filter(pl.col("len") > 1)["customer_id"].head(2)
)).sort(["customer_id", "valid_from"])
changed_sample


変更がなかった顧客は 1 行のまま保たれ、会員ランクやステータスが変化した顧客だけ `valid_to` と `is_current=False` が設定された旧バージョンと新バージョンの 2 行が保持されました（総行数 555 行、現行 525 行）。日次全件積み上げの 1,540 行と比べて大幅に重複を抑えつつ、時点履歴を完全に追跡できています。
